# 23-21 · Тестируем сканер и классификатор

Практика к разделу [«Проверяем сканирование и классификацию»](../../site/chapters/glava-23/23-24-testy-skanirovaniya.html). Использует настоящий пакет `safesort`.

## Цель

Написать и запустить тесты в духе `projects/python/safesort/tests/test_scanner.py` и `test_classifier.py`: вложенные файлы, исключение каталога результата, пустой каталог и известные/неизвестные расширения.

## Рабочий пример

In [1]:
import tempfile
from pathlib import Path

from safesort.config import Config, DEFAULT_EXTENSIONS
from safesort.scanner import scan
from safesort.classifier import classify


def test_scan_finds_nested_files(tmp_path):
    (tmp_path / "a").mkdir()
    (tmp_path / "a" / "otchet.pdf").write_text("...", encoding="utf-8")
    (tmp_path / "photo.jpg").write_text("...", encoding="utf-8")

    files = scan(tmp_path, Config())
    names = {f.path.name for f in files}
    assert names == {"otchet.pdf", "photo.jpg"}


def test_scan_skips_destination_directory(tmp_path):
    (tmp_path / "Sorted" / "documents").mkdir(parents=True)
    (tmp_path / "Sorted" / "documents" / "staryj.pdf").write_text("...", encoding="utf-8")

    files = scan(tmp_path, Config())
    assert files == []


def test_scan_empty_directory_returns_empty_list(tmp_path):
    files = scan(tmp_path, Config())
    assert files == []


def test_classify_known_extension():
    assert classify(".pdf", DEFAULT_EXTENSIONS) == "documents"


def test_classify_unknown_extension_is_other():
    assert classify(".xyz", DEFAULT_EXTENSIONS) == "other"


for test_func in (
    test_scan_finds_nested_files,
    test_scan_skips_destination_directory,
    test_scan_empty_directory_returns_empty_list,
):
    with tempfile.TemporaryDirectory() as tmp:
        test_func(Path(tmp))
    print(f"OK: {test_func.__name__}")

test_classify_known_extension()
test_classify_unknown_extension_is_other()
print("OK: test_classify_known_extension, test_classify_unknown_extension_is_other")

OK: test_scan_finds_nested_files
OK: test_scan_skips_destination_directory
OK: test_scan_empty_directory_returns_empty_list
OK: test_classify_known_extension, test_classify_unknown_extension_is_other


## Задание ★ Базовая практика

Напишите `test_scan_skips_symlinks(tmp_path)`, создающую символическую ссылку на файл, и убедитесь, что она не попадает в результат `scan()`.

In [2]:
def test_scan_skips_symlinks(tmp_path):
    fajl = tmp_path / "photo.jpg"
    fajl.write_text("...", encoding="utf-8")
    ssylka = tmp_path / "ssylka.jpg"
    try:
        ssylka.symlink_to(fajl)
    except (OSError, NotImplementedError):
        return  # символические ссылки не поддерживаются в этом окружении

    files = scan(tmp_path, Config())
    names = {f.path.name for f in files}
    assert "ssylka.jpg" not in names
    assert "photo.jpg" in names


with tempfile.TemporaryDirectory() as tmp:
    test_scan_skips_symlinks(Path(tmp))
print("OK: test_scan_skips_symlinks")

OK: test_scan_skips_symlinks
